# Online Retail — Raw Data Inspection and Exploratory Preparation

**Project:** E-commerce Sales Intelligence  
**Source:** [UCI Online Retail dataset](https://archive.ics.uci.edu/dataset/352/online+retail) (December 2010–December 2011)  
**Purpose:** Inspect data quality and investigate preparation decisions before final SQL analysis.

This notebook preserves the investigation of duplicates, missing values, cancellations, operational records, unusually large invoices, and preliminary revenue/customer calculations. Some cells construct **exploratory working datasets**; the final business KPIs are defined separately in [`04_SQL_Analysis`](../04_SQL_Analysis/).

### Run locally

1. Download the UCI source Excel file and place it at `02_Data/Raw/Online_Retail.xlsx` in the project root.
2. Install `pandas` and `openpyxl` in your Python environment.
3. Open the project in VS Code or Jupyter, then run cells in order. The path lookup below supports running from the project root, the `03_Python_Analysis` folder, or its containing `Analytics-Studio` folder.

The original data workbook is not included in the public repository. Saved outputs document the exploratory investigation; rerun the notebook from the beginning to reproduce them from the source workbook.

In [1]:
from pathlib import Path 

import pandas as pd

In [2]:
# Locate the UCI workbook across common VS Code/Jupyter working directories.
project_folder = "Project-01-Ecommerce-Sales-Intelligence"
raw_file = Path("02_Data") / "Raw" / "Online_Retail.xlsx"
working_dir = Path.cwd().resolve()
candidate_roots = [
    working_dir,
    working_dir.parent,
    working_dir / project_folder,
]
file_path = next(
    (root / raw_file for root in candidate_roots if (root / raw_file).is_file()),
    None,
)
if file_path is None:
    raise FileNotFoundError(
        "Online_Retail.xlsx not found. Download the UCI Online Retail dataset "
        "and place it in 02_Data/Raw/ in the project folder."
    )
print("Using source:", file_path)

Using source: C:\Users\grin-\OneDrive\Dokumente\Analytics-Studio\Project-01-Ecommerce-Sales-Intelligence\02_Data\Raw\Online_Retail.xlsx


In [3]:
excel_file = pd.ExcelFile(file_path)

In [4]:
excel_file.sheet_names

['Online Retail']

In [5]:
df_preview = pd.read_excel(
    file_path,
    sheet_name = 'Online Retail',
    nrows=10
)
df_preview

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850,United Kingdom
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01 08:26:00,4.25,17850,United Kingdom
7,536366,22633,HAND WARMER UNION JACK,6,2010-12-01 08:28:00,1.85,17850,United Kingdom
8,536366,22632,HAND WARMER RED POLKA DOT,6,2010-12-01 08:28:00,1.85,17850,United Kingdom
9,536367,84879,ASSORTED COLOUR BIRD ORNAMENT,32,2010-12-01 08:34:00,1.69,13047,United Kingdom


In [4]:
df = pd.read_excel(
    file_path,
    sheet_name = 'Online Retail'
)

## 1. Dataset structure and missing values

Start with the original source data; the raw DataFrame is kept unchanged as `df`.

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 40.0+ MB


In [9]:
df.shape


(541909, 8)

In [10]:
df.isna().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

## 2. Exact duplicate rows

Investigate identical rows before creating a separate deduplicated working copy.

In [11]:
df.duplicated().sum()

np.int64(5268)

In [12]:
df.loc[df.duplicated()].head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
517,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908.0,United Kingdom
527,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908.0,United Kingdom
537,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908.0,United Kingdom
539,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908.0,United Kingdom
555,536412,22327,ROUND SNACK BOXES SET OF 4 SKULLS,1,2010-12-01 11:49:00,2.95,17920.0,United Kingdom
587,536412,22273,FELTCRAFT DOLL MOLLY,1,2010-12-01 11:49:00,2.95,17920.0,United Kingdom
589,536412,22749,FELTCRAFT PRINCESS CHARLOTTE DOLL,1,2010-12-01 11:49:00,3.75,17920.0,United Kingdom
594,536412,22141,CHRISTMAS CRAFT TREE TOP ANGEL,1,2010-12-01 11:49:00,2.10,17920.0,United Kingdom
598,536412,21448,12 DAISY PEGS IN WOOD BOX,1,2010-12-01 11:49:00,1.65,17920.0,United Kingdom
600,536412,22569,FELTCRAFT CUSHION BUTTERFLY,2,2010-12-01 11:49:00,3.75,17920.0,United Kingdom


In [5]:
df[df["InvoiceNo"]==536409]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
483,536409,90199C,5 STRAND GLASS NECKLACE CRYSTAL,3,2010-12-01 11:45:00,6.35,17908.0,United Kingdom
484,536409,21479,WHITE SKULL HOT WATER BOTTLE,1,2010-12-01 11:45:00,3.75,17908.0,United Kingdom
485,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908.0,United Kingdom
486,536409,22785,SQUARECUSHION COVER PINK UNION FLAG,1,2010-12-01 11:45:00,6.75,17908.0,United Kingdom
487,536409,22975,SPACEBOY CHILDRENS EGG CUP,1,2010-12-01 11:45:00,1.25,17908.0,United Kingdom
488,536409,22972,CHILDREN'S SPACEBOY MUG,1,2010-12-01 11:45:00,1.65,17908.0,United Kingdom
489,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908.0,United Kingdom
490,536409,22568,FELTCRAFT CUSHION OWL,1,2010-12-01 11:45:00,3.75,17908.0,United Kingdom
491,536409,85116,BLACK CANDELABRA T-LIGHT HOLDER,1,2010-12-01 11:45:00,2.10,17908.0,United Kingdom
492,536409,22664,TOY TIDY DOLLY GIRL DESIGN,1,2010-12-01 11:45:00,2.10,17908.0,United Kingdom


In [5]:
df_clean = df.drop_duplicates()

In [7]:
df_clean.shape

(536641, 8)

## 3. Quantities and cancellation records

Negative quantities and `C`-prefixed invoices need separate investigation; they are not interchangeable definitions of customer returns.

In [8]:
df_clean["Quantity"].describe()

count    536641.000000
mean          9.620029
std         219.130156
min      -80995.000000
25%           1.000000
50%           3.000000
75%          10.000000
max       80995.000000
Name: Quantity, dtype: float64

In [9]:
df_clean["Quantity"].unique()

array([     6,      8,      2,     32,      3,      4,     24,     12,
           48,     18,     20,     36,     80,     64,     10,    120,
           96,     23,      5,      1,     -1,     50,     40,    100,
          192,    432,    144,    288,    -12,    -24,     16,      9,
          128,     25,     30,     28,      7,     56,     72,    200,
          600,    480,     -6,     14,     -2,     11,     33,     13,
           -4,     -5,     -7,     -3,     70,    252,     60,    216,
          384,    -10,     27,     15,     22,     19,     17,     21,
           34,     47,    108,     52,  -9360,    -38,     75,    270,
           42,    240,     90,    320,   1824,    204,     69,    -36,
         -192,   -144,    160,   2880,   1400,     39,    -48,    -50,
           26,   1440,     31,     82,     78,     97,     98,     35,
           57,    -20,    110,    -22,    -30,    -70,   -130,    -80,
         -120,    -40,    -25,    -14,    -15,    -69,   -140,   -320,
      

### Duplicate handling — finding and decision

The original workbook contained **541,909 rows**, including **5,268 exact duplicate rows**. Samples were examined for identical invoice, product, quantity, date, price, customer and country details. The duplicate rows were removed from the working copy, leaving **536,641 rows** in `df_clean`. The raw `df` remains unchanged.

In [13]:
df_clean[
    (df_clean["Quantity"] < 0) &
    (df_clean["UnitPrice"] == 0) &
    (df_clean["CustomerID"].notna())
]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country


In [7]:
df_clean[df_clean["InvoiceNo"].astype(str).str.startswith("C")]["Quantity"].describe()

count     9251.000000
mean       -29.787050
std       1147.997592
min     -80995.000000
25%         -6.000000
50%         -2.000000
75%         -1.000000
max         -1.000000
Name: Quantity, dtype: float64

In [8]:
df_clean.info()

<class 'pandas.DataFrame'>
Index: 536641 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    536641 non-null  object        
 1   StockCode    536641 non-null  object        
 2   Description  535187 non-null  object        
 3   Quantity     536641 non-null  int64         
 4   InvoiceDate  536641 non-null  datetime64[us]
 5   UnitPrice    536641 non-null  float64       
 6   CustomerID   401604 non-null  float64       
 7   Country      536641 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 43.8+ MB


### Negative quantities — interpretation and decision

After exact-duplicate removal, **9,251 rows** had invoice numbers beginning with `C`; all those rows had negative quantities. A further **1,336 negative-quantity rows** had invoice numbers that did not begin with `C`. Some of those non-`C` records use operational or inventory-adjustment descriptions.

**Decision:** Do not assume that every negative quantity is a customer return. Keep the original records in `df_clean` for investigation. Exclude `C`-prefixed invoices when creating the exploratory `df_orders` subset; use positive quantities when creating `df_revenue`.

## 4. Product descriptions, stock codes and unusual prices

Review adjustment-style descriptions and nonstandard transactions without automatically deleting every unusual record.

In [11]:
df_clean[df_clean["StockCode"]=="D"]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
141,C536379,D,Discount,-1,2010-12-01 09:41:00,27.50,14527.0,United Kingdom
9038,C537164,D,Discount,-1,2010-12-05 13:21:00,29.29,14527.0,United Kingdom
14498,C537597,D,Discount,-1,2010-12-07 12:34:00,281.00,15498.0,United Kingdom
19392,C537857,D,Discount,-1,2010-12-08 16:00:00,267.12,17340.0,United Kingdom
31134,C538897,D,Discount,-1,2010-12-15 09:14:00,5.76,16422.0,United Kingdom
...,...,...,...,...,...,...,...,...
479868,C577227,D,Discount,-1,2011-11-18 12:06:00,19.82,14527.0,United Kingdom
479869,C577227,D,Discount,-1,2011-11-18 12:06:00,16.76,14527.0,United Kingdom
493613,C578239,D,Discount,-1,2011-11-23 12:29:00,26.33,14912.0,Italy
516221,C579884,D,Discount,-1,2011-11-30 17:34:00,20.53,14527.0,United Kingdom


In [14]:
df_clean[df_clean["StockCode"]=="D"]["CustomerID"].nunique()

24

In [21]:
df_clean[df_clean["StockCode"].isin(["D"])]["Description"].unique()


array(['Discount'], dtype=object)

In [7]:
df_clean["Description"].value_counts().tail(30)


Description
wrongly marked carton 22804            1
SET 10 CARDS 3 WISE MEN 17107          1
FLOWER SHOP DESIGN MUG                 1
missing?                               1
SET OF 6 EASTER RAINBOW CHICKS         1
amazon adjust                          1
???lost                                1
dotcomstock                            1
SET 10 CARD CHRISTMAS STAMPS 16963     1
John Lewis                             1
sold with wrong barcode                1
CAT WITH SUNGLASSES BLANK CARD         1
dotcom adjust                          1
rusty thrown away                      1
check?                                 1
?? missing                             1
????missing                            1
CROCHET LILAC/RED BEAR KEYRING         1
LARGE HEART FLOWERS HOOK               1
SET 10 CARDS SNOWY ROBIN 17099         1
lost in space                          1
SET 10 CARDS SWIRLY XMAS TREE 17104    1
wet?                                   1
lost??                                 1
??? 

In [8]:
df_clean[df_clean["Description"].str.contains(
    "adjust|missing|lost|wet|damage|rusty|wrong|thrown",
    case=False,
    na=False
)][["StockCode", "Description", "Quantity", "UnitPrice", "InvoiceNo"]].head(30)

,StockCode,Description,Quantity,UnitPrice,InvoiceNo
13264,35833G,damages,-43,0.0,537432
42564,22501,reverse 21/5/10 adjustment,-100,0.0,540010
42566,22502,reverse 21/5/10 adjustment,-100,0.0,540012
50849,22617,"mouldy, thrown away.",-2600,0.0,540564
51049,72038P,damages,-990,0.0,540638
73589,85161,samples/damages,-20,0.0,542361
82794,84611B,thrown away,-1430,0.0,543257
82796,84612B,thrown away,-162,0.0,543259
82797,84613A,thrown away,-230,0.0,543260
82799,84614A,thrown away,-390,0.0,543262


In [13]:
df_clean[df_clean["StockCode"]==72732]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
131399,547560,72732,thrown away-can't sell,-2472,2011-03-23 17:28:00,0.0,NaN,United Kingdom


In [16]:
df_clean[df_clean["UnitPrice"]<0][
    ["StockCode", "Description", "Quantity", "InvoiceNo", "UnitPrice", "CustomerID"]
].head(30)

,StockCode,Description,Quantity,InvoiceNo,UnitPrice,CustomerID
299983,B,Adjust bad debt,1,A563186,-11062.06,NaN
299984,B,Adjust bad debt,1,A563187,-11062.06,NaN


In [24]:
df_clean[(df_clean["UnitPrice"]==0) & (df_clean["Description"].isna())]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.0,NaN,United Kingdom
1970,536545,21134,NaN,1,2010-12-01 14:32:00,0.0,NaN,United Kingdom
1971,536546,22145,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
1972,536547,37509,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
1987,536549,85226A,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom
...,...,...,...,...,...,...,...,...
535322,581199,84581,NaN,-2,2011-12-07 18:26:00,0.0,NaN,United Kingdom
535326,581203,23406,NaN,15,2011-12-07 18:31:00,0.0,NaN,United Kingdom
535332,581209,21620,NaN,6,2011-12-07 18:35:00,0.0,NaN,United Kingdom
536981,581234,72817,NaN,27,2011-12-08 10:33:00,0.0,NaN,United Kingdom


In [28]:
df_clean[
    (df_clean["UnitPrice"] == 0) &
    (df_clean["Description"].isna())
]["CustomerID"].notna().sum()

np.int64(0)

In [34]:
df_clean[
    (df_clean["UnitPrice"] == 0) &
    (df_clean["Description"].isna())
]["InvoiceNo"].head(20)

622     536414
1970    536545
1971    536546
1972    536547
1987    536549
1988    536550
2024    536552
2025    536553
2026    536554
2406    536589
4347    536764
4348    536765
7187    536995
7188    536996
7189    536997
7190    536998
7191    536999
7192    537000
7193    537001
7194    537002
Name: InvoiceNo, dtype: object

In [37]:
df_clean[
    (df_clean["UnitPrice"] == 0) &
    (df_clean["Description"].isna())
][["InvoiceNo", "InvoiceDate"]].head(20)

,InvoiceNo,InvoiceDate
622,536414,2010-12-01 11:52:00
1970,536545,2010-12-01 14:32:00
1971,536546,2010-12-01 14:33:00
1972,536547,2010-12-01 14:33:00
1987,536549,2010-12-01 14:34:00
1988,536550,2010-12-01 14:34:00
2024,536552,2010-12-01 14:34:00
2025,536553,2010-12-01 14:35:00
2026,536554,2010-12-01 14:35:00
2406,536589,2010-12-01 16:50:00


In [38]:
df_clean[df_clean["Description"].isna()]["StockCode"].value_counts().head(20)


StockCode
35965     10
23084     10
22084      9
22451      6
21784      5
22501      5
22866      5
23348      5
21067      5
21033      5
22139      4
84534B     4
22034      4
22114      4
21429      4
20966      4
22444      4
71477      4
21667      4
POST       4
Name: count, dtype: int64

In [20]:
df_clean["InvoiceNo"].astype(str).str.startswith("C").value_counts()


InvoiceNo
False    527390
True       9251
Name: count, dtype: int64

In [21]:
df_clean[
    (df_clean["Quantity"] < 0) &
    (~df_clean["InvoiceNo"].astype(str).str.startswith("C"))
]


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
2406,536589,21777,NaN,-10,2010-12-01 16:50:00,0.0,NaN,United Kingdom
4347,536764,84952C,NaN,-38,2010-12-02 14:42:00,0.0,NaN,United Kingdom
7188,536996,22712,NaN,-20,2010-12-03 15:30:00,0.0,NaN,United Kingdom
7189,536997,22028,NaN,-20,2010-12-03 15:30:00,0.0,NaN,United Kingdom
7190,536998,85067,NaN,-6,2010-12-03 15:30:00,0.0,NaN,United Kingdom
...,...,...,...,...,...,...,...,...
535333,581210,23395,check,-26,2011-12-07 18:36:00,0.0,NaN,United Kingdom
535335,581212,22578,lost,-1050,2011-12-07 18:38:00,0.0,NaN,United Kingdom
535336,581213,22576,check,-30,2011-12-07 18:38:00,0.0,NaN,United Kingdom
536908,581226,23090,missing,-338,2011-12-08 09:56:00,0.0,NaN,United Kingdom


## 5. Exploratory sales and order working copies

Create a separate working copy so investigative filters do not overwrite the deduplicated dataset.

In [6]:
df_sales = df_clean.copy()

In [7]:
df_sales["Revenue"] = df_sales["Quantity"] * df_sales["UnitPrice"]

In [12]:
df_sales[["Quantity", "UnitPrice", "Revenue"]].head(10)

,Quantity,UnitPrice,Revenue
0,6,2.55,15.30
1,6,3.39,20.34
2,8,2.75,22.00
3,6,3.39,20.34
4,6,3.39,20.34
5,2,7.65,15.30
6,6,4.25,25.50
7,6,1.85,11.10
8,6,1.85,11.10
9,32,1.69,54.08


### Examining invoice groups

An invoice may have multiple product lines, and some unusually large invoice groups require scrutiny before being treated as ordinary customer orders.

In [14]:
df_sales["InvoiceNo"].value_counts()

InvoiceNo
573585     1114
581219      749
581492      731
580729      721
558475      705
           ... 
581487        1
581491        1
C581499       1
581566        1
C581568       1
Name: count, Length: 25900, dtype: int64

In [15]:
df_sales[df_sales["InvoiceNo"]==573585]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue
430237,573585,11001,ASSTD DESIGN RACING CAR PEN,2,2011-10-31 14:41:00,3.29,NaN,United Kingdom,6.58
430238,573585,15036,ASSORTED COLOURS SILK FAN,15,2011-10-31 14:41:00,1.63,NaN,United Kingdom,24.45
430239,573585,15039,SANDALWOOD FAN,15,2011-10-31 14:41:00,1.63,NaN,United Kingdom,24.45
430240,573585,15056BL,EDWARDIAN PARASOL BLACK,1,2011-10-31 14:41:00,12.46,NaN,United Kingdom,12.46
430241,573585,15056N,EDWARDIAN PARASOL NATURAL,1,2011-10-31 14:41:00,12.46,NaN,United Kingdom,12.46
...,...,...,...,...,...,...,...,...,...
431346,573585,90209C,PINK ENAMEL+GLASS HAIR COMB,3,2011-10-31 14:41:00,2.07,NaN,United Kingdom,6.21
431347,573585,90214S,"LETTER ""S"" BLING KEY RING",1,2011-10-31 14:41:00,0.83,NaN,United Kingdom,0.83
431348,573585,DOT,DOTCOM POSTAGE,1,2011-10-31 14:41:00,2019.05,NaN,United Kingdom,2019.05
431349,573585,gift_0001_20,Dotcomgiftshop Gift Voucher £20.00,1,2011-10-31 14:41:00,16.67,NaN,United Kingdom,16.67


In [8]:
df_sales[df_sales["InvoiceNo"] == 581219]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
536087,581219,22580,ADVENT CALENDAR GINGHAM SACK,8,2011-12-08 09:28:00,4.13,NaN,United Kingdom
536088,581219,22581,WOOD STOCKING CHRISTMAS SCANDISPOT,10,2011-12-08 09:28:00,0.83,NaN,United Kingdom
536089,581219,22582,PACK OF 6 SWEETIE GIFT BOXES,1,2011-12-08 09:28:00,4.96,NaN,United Kingdom
536090,581219,22586,FELTCRAFT HAIRBAND PINK AND BLUE,1,2011-12-08 09:28:00,1.63,NaN,United Kingdom
536091,581219,22587,FELTCRAFT HAIRBAND RED AND BLUE,1,2011-12-08 09:28:00,1.63,NaN,United Kingdom
...,...,...,...,...,...,...,...,...
536831,581219,90206A,GOLD DIAMANTE STAR BROOCH,1,2011-12-08 09:28:00,4.98,NaN,United Kingdom
536832,581219,90206C,CRYSTAL DIAMANTE STAR BROOCH,2,2011-12-08 09:28:00,4.98,NaN,United Kingdom
536833,581219,90208,PAIR OF PINK FLOWER CLUSTER SLIDE,2,2011-12-08 09:28:00,2.48,NaN,United Kingdom
536834,581219,DOT,DOTCOM POSTAGE,1,2011-12-08 09:28:00,1008.96,NaN,United Kingdom


In [9]:
df_sales["InvoiceNo"].value_counts().tail(10)

InvoiceNo
C581463    1
C581470    1
581479     1
581483     1
C581484    1
581487     1
581491     1
C581499    1
581566     1
C581568    1
Name: count, dtype: int64

In [10]:
df_sales[df_sales["InvoiceNo"] == 581479]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
540389,581479,22087,PAPER BUNTING WHITE LACE,10,2011-12-09 09:00:00,2.95,17364.0,United Kingdom


In [8]:
# Exclude C-prefixed cancellation invoices from this exploratory order subset.
df_orders = df_sales[
    ~df_sales["InvoiceNo"].astype(str).str.startswith("C")
].copy()

In [14]:
df_orders.shape

(527390, 8)

In [15]:
df_orders["InvoiceNo"].astype(str).str.startswith("C").value_counts()

InvoiceNo
False    527390
Name: count, dtype: int64

In [16]:
df_orders["CustomerID"].notna().value_counts()

CustomerID
True     392732
False    134658
Name: count, dtype: int64

In [8]:
df_orders.groupby("InvoiceNo")["CustomerID"].count().value_counts().sort_index()

CustomerID
0      3528
1      1393
2       736
3       638
4       625
       ... 
435       1
442       1
529       1
533       1
542       1
Name: count, Length: 198, dtype: int64

In [13]:
df_orders["InvoiceNo"].nunique()

22064

## 6. Zero-price operational records

Identify clearly operational zero-price descriptions and remove only the identified subset from the exploratory `df_sales` working copy.

In [11]:
df_sales["UnitPrice"].eq(0).sum()

np.int64(2510)

In [12]:
df_sales.loc[df_sales["UnitPrice"] == 0, "Description"].value_counts().head(20)

Description
check                              159
?                                   47
damages                             45
damaged                             43
found                               25
sold as set on dotcom               20
adjustment                          16
Damaged                             14
FRENCH BLUE METAL DOOR SIGN 1        9
thrown away                          9
Unsaleable, destroyed.               9
amazon                               8
FRENCH BLUE METAL DOOR SIGN 8        8
Found                                8
FRENCH BLUE METAL DOOR SIGN 4        7
FRENCH BLUE METAL DOOR SIGN No       7
OWL DOORSTOP                         7
FRENCH BLUE METAL DOOR SIGN 3        7
RECIPE BOX PANTRY YELLOW DESIGN      7
??                                   7
Name: count, dtype: int64

In [10]:
operational_keywords = (
    "check|damage|damaged|damages|adjustment|thrown away|"
    "unsaleable|destroyed|lost|missing|smashed"
)

In [11]:
df_sales[
    df_sales["UnitPrice"].eq(0) &
    df_sales["Description"].str.contains(
        operational_keywords,
        case=False,
        na=False
    )
].shape[0]

368

In [12]:
df_sales.loc[
    df_sales["UnitPrice"].eq(0) &
    df_sales["Description"].str.contains(
        operational_keywords,
        case=False,
        na=False
    ),
    "Description"
].value_counts()

Description
check                            159
damages                           45
damaged                           43
adjustment                        16
Damaged                           14
thrown away                        9
Unsaleable, destroyed.             9
damages?                           5
wet damaged                        5
smashed                            4
missing                            3
CHECK                              3
reverse 21/5/10 adjustment         2
Adjustment                         2
printing smudges/thrown away       2
?missing                           2
stock check                        2
damages wax                        2
???missing                         2
mouldy, thrown away.               1
samples/damages                    1
incorrectly made-thrown away.      1
Thrown away.                       1
damages/display                    1
damaged stock                      1
?lost                              1
thrown away-can't sell.   

In [13]:
operational_zero_price = (
    df_sales["UnitPrice"].eq(0) &
    df_sales["Description"].str.contains(
        operational_keywords,
        case=False,
        na=False
    )
)

In [14]:
df_sales = df_sales[~operational_zero_price].copy()

## 7. Missing customer identifiers and unusually large invoice groups

Inspect invoice-level identifiers. The subsequent exploratory exclusion selects invoice groups with at least 500 lines and no recorded customer identifier; it is not a general rule for all missing customer IDs.

In [16]:
df_orders.groupby("InvoiceNo")["CustomerID"].nunique().loc[lambda x: x > 1]

Series([], Name: CustomerID, dtype: int64)

In [19]:
recoverable = df_orders[
    df_orders["CustomerID"].isna() &
    df_orders.groupby("InvoiceNo")["CustomerID"].transform("count").gt(0)
]

recoverable.shape

(0, 9)

In [16]:
order_summary = (
    df_orders.groupby("InvoiceNo")
    .agg(
        line_count=("InvoiceNo", "size"),
        customer_count=("CustomerID", "count")
    )
    .sort_values("line_count", ascending=False)
)

order_summary.head(50)

,line_count,customer_count
InvoiceNo,,
573585,1114,0
581219,749,0
581492,731,0
580729,721,0
558475,705,0
579777,687,0
581217,676,0
537434,675,0
580730,662,0


In [22]:
order_summary[order_summary["customer_count"] == 0].tail(20)

,line_count,customer_count
InvoiceNo,,
552178,1,0
573293,1,0
552187,1,0
552214,1,0
551900,1,0
552215,1,0
552216,1,0
552218,1,0
552219,1,0


In [17]:
order_summary[
    (order_summary["line_count"] >= 500) &
    (order_summary["customer_count"] == 0)
].shape

(46, 2)

In [18]:
df_sales["Revenue"].sum()

np.float64(9726006.954)

In [19]:
df_orders["Revenue"].sum()

np.float64(10619986.684)

### Decision on unusually large invoice groups

The invoice-level investigation identified **46 invoice groups** containing at least 500 lines each and no non-null `CustomerID` values. These were excluded from the exploratory `clean_base` working dataset because they differed from typical identifiable customer transactions. This is an **analysis-specific rule**, not evidence that every missing customer ID makes a transaction invalid.

In [20]:
large_no_customer = order_summary[
    (order_summary["line_count"] >= 500) &
    (order_summary["customer_count"] == 0)
].index

clean_base = df_sales[
    ~df_sales["InvoiceNo"].isin(large_no_customer)
].copy()

In [21]:
df_sales = clean_base.copy()

In [24]:
df_orders = clean_base[
    ~clean_base["InvoiceNo"].astype(str).str.startswith("C")
].copy()

### Exploratory dataset lineage

```text
Original df
    └── df_clean       remove exact duplicate rows
         └── df_sales  add Revenue; remove selected operational zero-price rows
              └── clean_base  exclude 46 unusually large anonymous invoice groups
                   ├── df_sales   retained working copy
                   └── df_orders  exclude C-prefixed invoices
                        └── df_revenue  keep positive-quantity lines only
```

These intermediate names and steps describe the investigation in this notebook. The published dashboard's metric definitions additionally depend on the final BigQuery SQL filters.

## 8. Intermediate working-dataset checks

These counts describe the **exploratory** subsets after the targeted invoice-group exclusion; they are not final Tableau KPI totals.

In [23]:
clean_base.shape

(508641, 9)

In [32]:
df_sales.shape

(508641, 9)

In [25]:
df_orders.shape

(499390, 9)

In [34]:
df_orders["InvoiceNo"].astype(str).str.startswith("C").sum()

np.int64(0)

In [26]:
df_orders["InvoiceNo"].isin(large_no_customer).sum()

np.int64(0)

In [27]:
df_orders["InvoiceNo"].nunique()

21650

In [30]:
df_orders["InvoiceNo"].value_counts().describe()

count    21650.000000
mean        23.066513
std         37.149662
min          1.000000
25%          4.000000
50%         14.000000
75%         27.000000
max        542.000000
Name: count, dtype: float64

In [35]:
df_orders["Country"].value_counts()

Country
United Kingdom          454479
Germany                   9027
France                    8393
EIRE                      7883
Spain                     2480
Netherlands               2363
Belgium                   2031
Switzerland               1959
Portugal                  1492
Australia                 1184
Norway                    1072
Italy                      758
Channel Islands            747
Finland                    685
Cyprus                     603
Sweden                     450
Unspecified                442
Austria                    398
Denmark                    380
Poland                     330
Japan                      321
Israel                     292
Hong Kong                  280
Singapore                  222
Iceland                    182
USA                        179
Canada                     151
Greece                     145
Malta                      112
United Arab Emirates        68
European Community          60
RSA                         58


In [36]:
df_orders["Quantity"].describe()

count    499390.000000
mean         10.939775
std         162.450272
min       -5368.000000
25%           1.000000
50%           4.000000
75%          12.000000
max       80995.000000
Name: Quantity, dtype: float64

In [37]:
df_orders["Revenue"].describe()


count    499390.000000
mean         20.682083
std         279.281587
min      -11062.060000
25%           3.950000
50%          10.000000
75%          17.700000
max      168469.600000
Name: Revenue, dtype: float64

## 9. Positive-quantity exploratory revenue

Construct a positive-quantity subset from `df_orders`, while retaining other transaction types in earlier working datasets for transparency.

In [53]:
(df_orders["Quantity"] < 0).value_counts()

Quantity
False    498364
True       1026
Name: count, dtype: int64

In [45]:
df_revenue = df_orders[df_orders["Quantity"] > 0].copy()

### Revenue-subset definition

For this exploratory revenue subset, `df_revenue` retains **positive-quantity** lines from `df_orders`. Negative quantities may reflect cancellations, reversals, or stock adjustments; excluding them here avoids treating them as ordinary sales. That decision does **not** constitute a full refund reconciliation or a final net-revenue measure.

The following revenue and customer summaries are **intermediate exploratory results**, not the final SQL/Tableau KPIs.

In [46]:
df_revenue.shape

(498364, 9)

In [54]:
df_revenue["Revenue"].sum()

np.float64(10328425.404000001)

In [57]:
(df_orders["UnitPrice"] == 0).sum()

np.int64(2141)

In [58]:
(clean_base["UnitPrice"] == 0).sum()

np.int64(2141)

In [60]:
clean_base[clean_base["UnitPrice"] <= 0]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.0,NaN,United Kingdom,0.0
1970,536545,21134,NaN,1,2010-12-01 14:32:00,0.0,NaN,United Kingdom,0.0
1971,536546,22145,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom,0.0
1972,536547,37509,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom,0.0
1987,536549,85226A,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom,0.0
...,...,...,...,...,...,...,...,...,...
535332,581209,21620,NaN,6,2011-12-07 18:35:00,0.0,NaN,United Kingdom,0.0
536981,581234,72817,NaN,27,2011-12-08 10:33:00,0.0,NaN,United Kingdom,0.0
538504,581406,46000M,POLYESTER FILLER PAD 45x45cm,240,2011-12-08 13:58:00,0.0,NaN,United Kingdom,0.0
538505,581406,46000S,POLYESTER FILLER PAD 40x40cm,300,2011-12-08 13:58:00,0.0,NaN,United Kingdom,0.0


In [62]:
df_orders["UnitPrice"].le(0).sum()

np.int64(2143)

In [63]:
operational_zero_price = (
    df_sales["UnitPrice"].eq(0) &
    df_sales["Description"].str.contains(
        "check|damage|damaged|damages|adjustment|thrown away|unsaleable|destroyed|lost|missing|smashed",
        case=False,
        na=False
    )
)

operational_zero_price.sum()

np.int64(0)

In [65]:
df_orders["UnitPrice"].eq(0).sum()

np.int64(2141)

## 10. Customer and nonstandard stock-code checks

These outputs are exploratory cross-checks. Stock codes reviewed below are **not automatically excluded** by this notebook.

In [66]:
customer_revenue = (
    df_revenue[df_revenue["CustomerID"].notna()]
    .groupby("CustomerID", as_index=False)["Revenue"]
    .sum()
    .sort_values("Revenue", ascending=False)
)
customer_revenue.head(10)

,CustomerID,Revenue
1690,14646.0,280206.02
4202,18102.0,259657.30
3729,17450.0,194390.79
3009,16446.0,168472.50
1880,14911.0,143711.17
55,12415.0,124914.53
1334,14156.0,117210.08
3772,17511.0,91062.38
2703,16029.0,80850.84
0,12346.0,77183.60


In [67]:
top_customer = df_revenue[df_revenue["CustomerID"] == 14646]

top_customer_summary = {
    "Total Revenue": top_customer["Revenue"].sum(),
    "Number of Orders": top_customer["InvoiceNo"].nunique(),
    "Total Items": top_customer["Quantity"].sum()
}

top_customer_summary

{'Total Revenue': np.float64(280206.02),
 'Number of Orders': 74,
 'Total Items': np.int64(197491)}

In [68]:
df_revenue.isna().sum()


InvoiceNo           0
StockCode           0
Description       592
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     105632
Country             0
Revenue             0
dtype: int64

In [69]:
df_revenue["Description"].isna().sum()

np.int64(592)

In [70]:
df_revenue.loc[df_revenue["Description"].isna(), "Revenue"].sum()

np.float64(0.0)

In [72]:
df_revenue.duplicated().sum()

np.int64(0)

In [73]:
df_revenue[df_revenue["UnitPrice"] < 0]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue
299983,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom,-11062.06
299984,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom,-11062.06


In [74]:
df_revenue[df_revenue["StockCode"] == "B"]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue
299982,A563185,B,Adjust bad debt,1,2011-08-12 14:50:00,11062.06,NaN,United Kingdom,11062.06
299983,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom,-11062.06
299984,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom,-11062.06


In [75]:
df_revenue[
    df_revenue["Description"]
    .str.contains("bad debt", case=False, na=False)
]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue
299982,A563185,B,Adjust bad debt,1,2011-08-12 14:50:00,11062.06,NaN,United Kingdom,11062.06
299983,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom,-11062.06
299984,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom,-11062.06


In [76]:
df_revenue[
    df_revenue["StockCode"].str.fullmatch(r"[A-Za-z]", na=False)
]["StockCode"].value_counts()

StockCode
M    322
B      3
S      2
m      1
Name: count, dtype: int64

In [77]:
df_revenue[
    df_revenue["StockCode"].isin(["M", "B", "S", "m"])
][["StockCode", "Description", "Quantity", "UnitPrice", "Revenue"]]


,StockCode,Description,Quantity,UnitPrice,Revenue
2239,M,Manual,1,1.25,1.25
2250,M,Manual,1,18.95,18.95
5684,M,Manual,1,2.55,2.55
6798,M,Manual,2,0.85,1.70
7976,M,Manual,12,0.42,5.04
...,...,...,...,...,...
526015,M,Manual,1,219.50,219.50
526018,M,Manual,800,0.25,200.00
530876,M,Manual,1,0.85,0.85
531432,M,Manual,4,1.25,5.00


In [78]:
df_revenue[
    df_revenue["StockCode"].isin(["B", "S", "m"])
][["StockCode", "Description", "Quantity", "UnitPrice", "Revenue"]]

,StockCode,Description,Quantity,UnitPrice,Revenue
40383,m,Manual,1,2.55,2.55
152709,S,SAMPLES,1,30.00,30.00
299982,B,Adjust bad debt,1,11062.06,11062.06
299983,B,Adjust bad debt,1,-11062.06,-11062.06
299984,B,Adjust bad debt,1,-11062.06,-11062.06
419666,S,SAMPLES,1,33.05,33.05


In [79]:
df_revenue[
    df_revenue["StockCode"].isin(["B", "S"])
]["Revenue"].sum()

np.float64(-10999.01)

## 11. Summary of preparation decisions and analytical boundaries

| Investigation or rule | Working dataset | What this notebook actually does |
|---|---|---|
| Exact duplicate rows | `df_clean` | Removes 5,268 duplicates; preserves raw `df` |
| Missing `CustomerID` | `df_clean` and downstream | Does not automatically remove all rows with missing IDs |
| Operational zero-price descriptions | Initial `df_sales` | Excludes 368 identified records |
| Unusually large anonymous invoices | `clean_base` | Excludes 46 invoice groups with ≥500 lines and no customer identifier |
| Cancellation invoices | `df_orders` | Excludes invoices with numbers beginning `C` |
| Positive quantities | `df_revenue` | Retains only `Quantity > 0` |
| Nonstandard stock codes (`M`, `m`, `B`, `S`) | Investigation only | **Inspected here, not excluded by the displayed Python code** |

**Important interpretation:** The saved exploratory `df_revenue` sum and customer totals above are **not** the final published business metrics. The BigQuery scripts apply the project-specific filters (including cancellation, quantity, and selected stock-code exclusions), and the Tableau dashboard reports the corresponding analyzed results. Compare figures only when their dataset scope and filters match.

**Limitations:** Missing customer identifiers constrain customer-level analysis; unusually large invoice groups were excluded under an explicit judgment rule; some stock codes represent adjustments rather than merchandise; and excluding cancellation/negative lines does not fully reconstruct net revenue after returns. The dataset ends on **December 9, 2011**, so December is incomplete.